# Creating the document term matrix -- Approach 2: spaCy (guided, deterministic)

## Why this notebook exists

The naive-LLM notebook showed two concrete failure modes: a bare prompt
returns free-form prose mixed with English commentary and markdown (not
tokens), and even a well-behaved response still leaves stopwords
undecided and dominating the raw counts. This notebook builds the same
kind of output -- a list of normalized, content-bearing tokens per
document -- using a deterministic, versioned, inspectable pipeline
instead of an API call. Every decision here (which stopword list, which
POS tags count as "content," how multi-word entities are joined) is
explicit in code and can be cited in a methods section.

We pick up from the same starting point as the LLM notebook: `text_clean`
from `CleanCorpus.csv`, not the raw PDF text.


In [ ]:
import pandas as pd

corpus = pd.read_csv("CleanCorpus.csv")
corpus.info()
corpus[["party", "n_words_clean"]]


## Setup: spaCy and the Spanish pipeline

Run the download line once. `es_core_news_lg` gives the most accurate
tagger/lemmatizer/NER of the three Spanish pipeline sizes (`sm`, `md`,
`lg`) at the cost of a larger download -- worth it here since accuracy
of lemmatization and entity recognition is the whole point of this
notebook. If class time or bandwidth is tight, `es_core_news_md` is a
reasonable fallback; avoid `sm` for this task specifically, since its
smaller tagger is noticeably less reliable at exactly the POS and NER
decisions this pipeline depends on.


In [ ]:
# Run once (comment out after the first run in a given environment):
# !python -m spacy download es_core_news_lg

import spacy

nlp = spacy.load("es_core_news_lg")


## Mapping the original instructions onto spaCy

Back at the start of this project, the plan was a single LLM prompt with
10 numbered instructions. Here is where each one is actually implemented,
concretely, in code:

| # | Instruction | spaCy mechanism |
|---|---|---|
| 1 | Lowercase | `token.lemma_.lower()` at the end of the content-word branch |
| 2 | Remove punctuation | `token.is_punct` |
| 3 | Remove stopwords | `token.is_stop` (spaCy's built-in Spanish list -- citable, versioned) |
| 4 | Verbs to infinitive | `token.lemma_` (spaCy's lemmatizer does this natively) |
| 5 | Plural nouns to singular | `token.lemma_` (same mechanism as #4) |
| 6 | Preserve proper names/acronyms | `token.pos_ == "PROPN"` and an uppercase-text check |
| 7 | Preserve years, discard other numbers | a `\d{4}` regex check before the generic `like_num` filter |
| 8 | Multi-word entities with underscores | `doc.retokenize()` merging `doc.ents` spans before filtering |
| 9 | Keep only content-word POS | explicit `{"NOUN","VERB","ADJ","ADV"}` set |
| 10 | Return a clean token list | the function returns a Python list directly -- no JSON parsing needed |


## The pipeline function

One thing this pipeline gets right that's easy to get wrong: multi-word
entities are merged into a single token *before* any other filter runs.
Testing this against a synthetic example ("Estados Unidos") surfaced a
real edge case worth knowing about -- "Estados" on its own is flagged as
a Spanish stopword by spaCy's static word list, since it's also the
plural of "estado" (state, as in "estado de salud"). If entity merging
happened *after* stopword filtering instead of before, half of "Estados
Unidos" would silently disappear. Merging first, and explicitly setting
the merged token's LEMMA and POS rather than letting it inherit
whatever the first word in the span happened to have, avoids that.


In [ ]:
import re

def lemmatize_spacy_doc(doc):
    """
    Normalize one already-processed spaCy Doc into a list of lexical content
    tokens, following the same intent as the original 10-instruction LLM
    prompt, but with each decision made explicitly and deterministically in
    code. Takes a Doc (not raw text) so it can be reused efficiently with
    nlp.pipe() when processing many documents -- see the batch cell below.
    """
    # Merge multi-word named entities (instruction 8) into single tokens
    # FIRST, before any other filter sees their individual words. Explicitly
    # setting LEMMA (underscore-joined, lowercase) and POS (PROPN) instead of
    # relying on defaults, since an unmerged constituent word can carry a
    # misleading is_stop / POS value that would otherwise apply to the merge.
    with doc.retokenize() as retokenizer:
        for ent in doc.ents:
            if len(ent) > 1:
                merged_lemma = "_".join(t.text for t in ent).lower()
                retokenizer.merge(ent, attrs={"LEMMA": merged_lemma, "POS": "PROPN"})

    tokens = []
    for token in doc:
        # Any entity (multi-word, now merged above, or a single-token
        # entity like a lone country name) bypasses every filter below and
        # is preserved as-is -- this is instruction 6 + instruction 8
        # together, handled before instruction 9's POS filter would
        # otherwise have a chance to exclude it.
        if token.ent_type_:
            tokens.append(token.lemma_ if "_" in token.lemma_ else token.text)
            continue

        # Instruction 7: preserve four-digit years specifically, discard
        # every other numeric token (page-reference numbers, statistics,
        # budget figures written as digits).
        if re.fullmatch(r"\d{4}", token.text):
            tokens.append(token.text)
            continue
        if token.like_num or token.pos_ == "NUM":
            continue

        # Instructions 2 and 3: punctuation and stopwords. token.is_stop
        # uses spaCy's built-in, versioned Spanish stopword list -- citable
        # in a methods section, unlike an LLM's undocumented internal choice.
        if token.is_punct or token.is_space or token.is_stop:
            continue

        # Instruction 6: proper nouns and acronyms not already caught by
        # NER above. Preserve the original text and casing -- do NOT
        # lemmatize or lowercase a proper noun or an acronym like "OCDE".
        if token.pos_ == "PROPN" or (token.text.isupper() and len(token.text) > 1):
            tokens.append(token.text)
            continue

        # Instruction 9: keep only nouns, verbs, adjectives, and adverbs.
        # Everything else (determiners, prepositions, conjunctions not
        # already caught by is_stop, auxiliary particles) is dropped here.
        if token.pos_ not in {"NOUN", "VERB", "ADJ", "ADV"}:
            continue

        # Instructions 1, 4, 5: lowercase + lemmatize (spaCy's lemmatizer
        # converts verbs to infinitive and plural nouns to singular as part
        # of the same .lemma_ attribute -- one mechanism handles both).
        tokens.append(token.lemma_.lower())

    return tokens


def lemmatize_spacy(text):
    """Convenience wrapper for a single string -- runs the pipeline, then
    delegates to lemmatize_spacy_doc(). For processing many documents, use
    nlp.pipe() with lemmatize_spacy_doc() directly instead (see below) to
    avoid running the pipeline once per call in a plain Python loop."""
    return lemmatize_spacy_doc(nlp(text))


## Same toy example as the LLM notebook, for direct comparison

Compare this output to the markdown-and-commentary response the bare LLM
prompt produced for the same sentence.


In [ ]:
sample_tokens = lemmatize_spacy("Los niños están corriendo rápidamente.")
print(sample_tokens)


No parsing step needed -- the function returns a Python list directly.
Same input every time produces the same output every time: no
temperature, no model-version drift, no risk of the response changing
shape between runs.


## Run across all six documents

spaCy's `.pipe()` processes documents in a batch more efficiently than
calling `nlp()` in a plain loop, and disabling the dependency parser
(not used by this pipeline -- only the tagger, lemmatizer, and NER are
needed) speeds things up further on these longer documents.


In [ ]:
# nlp.pipe() processes the 6 documents as a batch (more efficient than
# calling nlp() in a plain loop), and disable=["parser"] skips dependency
# parsing -- lemmatize_spacy_doc() never reads token.dep_ or token.head, so
# this is pure speed, it doesn't change the output. Feeding each resulting
# Doc straight into lemmatize_spacy_doc() avoids running the pipeline a
# second time per document, which calling lemmatize_spacy(text) in a loop
# would do.
spacy_tokens = {}
for party, doc in zip(corpus["party"], nlp.pipe(corpus["text_clean"], disable=["parser"])):
    spacy_tokens[party] = lemmatize_spacy_doc(doc)

for party, toks in spacy_tokens.items():
    print(f"{party}: {len(toks)} tokens")


## Building the DTM

The tokens are already produced (unlike the naive-LLM notebook, there's
no free text to re-tokenize) -- `CountVectorizer` just needs to count
them. Passing `analyzer=lambda tokens: tokens` tells it to use each list
as-is rather than trying to tokenize a string.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_spacy = CountVectorizer(analyzer=lambda tokens: tokens)
dtm_spacy = vectorizer_spacy.fit_transform(
    [spacy_tokens[party] for party in corpus["party"]]
)
dtm_spacy_df = pd.DataFrame(
    dtm_spacy.toarray(),
    index=corpus["party"],
    columns=vectorizer_spacy.get_feature_names_out(),
)
dtm_spacy_df


Same diagnostic as the naive-LLM notebook, for a direct comparison of
vocabulary size and sparsity between the two approaches.


In [ ]:
n_docs, n_terms = dtm_spacy_df.shape
n_nonzero = (dtm_spacy_df > 0).values.sum()
sparsity = 1 - (n_nonzero / (n_docs * n_terms))

print(f"Vocabulary size (columns): {n_terms}")
print(f"Sparsity: {sparsity:.2%} of cells are zero")
dtm_spacy_df.sum(axis=0).sort_values(ascending=False).head(20)


## TF-IDF, derived from the same counts

Same reasoning as the LLM notebook: TF-IDF is a reweighting of the counts
already computed, not a separate pipeline. Keep the raw-count DTM as the
canonical artifact for anything that needs actual counts (Wordfish,
Wordscores, LDA, keyness statistics); use this TF-IDF version only for
similarity/clustering-style comparisons.


In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer

tfidf_transformer = TfidfTransformer()
tfidf_spacy = tfidf_transformer.fit_transform(dtm_spacy)
tfidf_spacy_df = pd.DataFrame(
    tfidf_spacy.toarray(),
    index=dtm_spacy_df.index,
    columns=dtm_spacy_df.columns,
)
tfidf_spacy_df


## Save, and compare against the naive-LLM DTM if it's already been run


In [ ]:
dtm_spacy_df.to_csv("DTM_spacy_counts.csv")
tfidf_spacy_df.to_csv("DTM_spacy_tfidf.csv")
print("Saved: DTM_spacy_counts.csv, DTM_spacy_tfidf.csv")


In [ ]:
import os

if os.path.exists("DTM_llm_naive_counts.csv"):
    dtm_llm_naive_df = pd.read_csv("DTM_llm_naive_counts.csv", index_col=0)
    comparison = pd.DataFrame({
        "vocab_size": [dtm_llm_naive_df.shape[1], dtm_spacy_df.shape[1]],
        "sparsity_pct": [
            round(100 * (1 - (dtm_llm_naive_df.values > 0).sum() / dtm_llm_naive_df.size), 1),
            round(100 * (1 - (dtm_spacy_df.values > 0).sum() / dtm_spacy_df.size), 1),
        ],
    }, index=["naive_llm", "spacy"])
    print(comparison)
else:
    print("DTM_llm_naive_counts.csv not found yet -- run the naive-LLM notebook first to compare.")
